## Entregável 1

In [ ]:
import mne
import matplotlib.pyplot as plt

In [ ]:
raw = mne.io.read_raw_edf('n1.edf', preload=False)

In [ ]:
print(f'canais encontrados: {len(raw.ch_names)} => {raw.ch_names}')

In [ ]:
print(f'Frequência de amostragem (fs): {raw.info['sfreq']} hz')

### Guia de sinais

In [ ]:
import pandas as pd

data = {
    "Sinal": ["C4-A1 / F4-C4", "ROC-LOC", "EMG1-EMG2", "ECG1-ECG2", "TERMISTORE", "TORACE / ADDOME", "SpO2"],
    "Tipo": ["EEG (Cérebro)", "EOG (Olhos)", "EMG (Músculo)", "ECG (Coração)", "Fluxo Aéreo", "Esforço Resp.", "Oxímetro"],
    "Por que é importante?": [
        "Identifica estágios de sono e o CAP (ondas Delta, fusos, complexos K).",
        "Essencial para o sono REM (movimentos rápidos dos olhos).",
        "Mostra o relaxamento muscular (atonia no REM) ou bruxismo.",
        "Mede frequência cardíaca e ajuda a identificar despertares.",
        "Verifica se o paciente está respirando pelo nariz/boca.",
        "Mede o movimento do peito e abdômen (detecta apneia).",
        "Nível de oxigênio no sangue (ajuda a validar eventos respiratórios)."
    ],
    "Detalhe Técnico": ["512 Hz / Baixa voltagem", "512 Hz / Capta dipolos oculares", "Alta frequência / Ruído muscular", "Sinal forte / Picos R-R", "Sinal lento", "Sinal muito lento / Sensores de cinta", "Valor percentual constante"]
}

df_sinais = pd.DataFrame(data)
print(df_sinais.to_string(index=False))

### Plots

In [ ]:
# Escolhendo canais representativos: 1 de olho, 2 de cérebro, 1 de músculo, 1 de coração
canais_selecionados = ['ROC-LOC', 'C4-A1', 'F4-C4', 'EMG1-EMG2', 'ECG1-ECG2']

# Plotando com os canais que escolhi
raw.plot(order=[raw.ch_names.index(c) for c in canais_selecionados], 
         duration=2, 
         title='Sinais Selecionados para Análise de Integridade',
         scalings='auto')

In [ ]:
# Plotando um trecho estabilizado (20 a 22 segundos)
raw.plot(start=20, 
         duration=2, 
         order=[raw.ch_names.index(c) for c in canais_selecionados], 
         scalings='auto', 
         title='Sinal Estabilizado (20s-22s)')

In [ ]:
# OBJETIVO: Garantir a integridade física e fisiológica do sinal
# DATASET: CAP Sleep Database (n1.edf / n1.txt)

# 1. TAXA DE AMOSTRAGEM (fs): 
# Identificada como 512 Hz. Justificada pelo Critério de Nyquist: 
# Como as frequências de interesse no sono (EEG) chegam a ~100Hz, 
# a fs deve ser > 200Hz. 512Hz garante que não haja Aliasing.

# 2. PROTOCOLO EXPERIMENTAL: 
# Polissonografia noturna completa (PSG) realizada em ambiente clínico 
# controlado (Ospedale Maggiore di Parma)

# 3. IDENTIFICAÇÕES VISUAIS REALIZADAS:
# - DRIFT (Deriva): Observada nos primeiros 0.5s. Instabilidade na linha 
#   de base (baseline) enquanto o hardware estabiliza.
# - SATURAÇÃO: Identificada visualmente em canais de maior amplitude (EMG/ECG) 
#   onde o sinal "achata" nos limites de voltagem.
# - RUÍDO: Presença de ruído de alta frequência e possível ruído de 
#   linha (50/60 Hz) que exigirá filtragem posterior.

# 4. CANAIS SELECIONADOS:
# Focamos em EEG (C4-A1), EOG (ROC-LOC), EMG e ECG para capturar 
# a microestrutura do sono e validar a qualidade do sinal.

# Entregável 2

In [ ]:
import numpy as np 
from scipy.stats import kurtosis, skew
import antropy as ant

In [ ]:
def decisor(kurt, se):
    # 1. Critério de Outlier Instrumental (Picos de movimento/eletrodo)
    if kurt > 10: 
        return "Rejeitar", "Outlier Instrumental (Pico)"
    
    # 2. Critério de Qualidade Espectral (Sinal muito caótico ou morto)
    if se < 0.1 or se > 0.9:
        return "Rejeitar", "Baixa Qualidade Espectral"
    
    # Se passou nos dois, aceitamos para a análise estatística
    return "Aceitar", "Fisiológico"

In [ ]:
def percorre_metrifica_decide(total_janelas, num_amostras, sfreq, janelas_limpas):
    for i in range(total_janelas):
        inicio = i* num_amostras
        fim = inicio + num_amostras
        janela = data[inicio:fim]

        kurt = kurtosis(janela)
        sk = skew(janela)    
        spec_entropy = ant.spectral_entropy(janela, sf=sfreq, method='welch', normalize=True)
        # SNR estimado: RMS do sinal dividido pelo desvio padrão do ruído estimado
        # (Aqui, vamos considerar que o ruído é a variação de alta frequência)
        # Uma forma simples é comparar a amplitude pico-a-pico com a variância
        rms = np.sqrt(np.mean(janela**2))
        std = np.std(janela)
        snr = 20 * np.log10(rms / std) if std > 0 else 0
        status, motivo = decisor(kurt, spec_entropy)
        if status == "Aceitar": janelas_limpas.append(janela)
        print(f"Janela {i+1} | Kurtosis: {kurt:.2f} | Skewness: {sk:.2f} | Entropia: {spec_entropy:.2f} | SNR: {snr:.2f} dB | {status} - {motivo}")


In [ ]:
duracao_janela = 30 #30 segundos
sfreq = raw.info['sfreq']
num_amostras = int(duracao_janela * sfreq)

data, times = raw.get_data(picks='C4-A1', return_times = True)
data = data[0]

total_janelas = int(len(data) / num_amostras)

janelas_limpas = []

percorre_metrifica_decide(total_janelas, num_amostras, sfreq, janelas_limpas)

base_estatistica = np.array(janelas_limpas)

In [ ]:
np.shape(base_estatistica)

In [ ]:
# =============================================================================
# ENTREGÁVEL 2 - AVALIAÇÃO DA QUALIDADE DO SINAL (SQI)
# =============================================================================

# OBJETIVO: 
# Garantir a integridade fisiológica dos dados que alimentarão o futuro modelo 
# preditivo, eliminando segmentos (janelas de 30s) contaminados por falhas de 
# hardware, perda de contato ou movimentos bruscos do paciente.

# METODOLOGIA DE DECISÃO (THRESHOLDS AUTOMÁTICOS):
# - Kurtosis (> 10): Aplicada para detecção de Outliers Instrumentais. Picos 
#   súbitos de voltagem (ex: batidas no eletrodo) geram distribuições com 
#   curtose elevada. Segmentos com esse comportamento foram rejeitados.
# - Entropia Espectral (< 0.1 ou > 0.9): Utilizada para avaliar a complexidade 
#   do sinal. Valores baixos indicam sinal saturado ("preso"), enquanto valores 
#   próximos a 1 indicam ruído branco puro (canal desconectado).
# - SNR (Razão Sinal-Ruído): Calculada, porém observou-se que no sinal bruto 
#   de EEG o SNR tende a 0 dB devido ao forte DC Offset e ruído de baixa 
#   frequência. Sua aplicação como critério de corte rigoroso foi postergada 
#   para após a etapa de filtragem digital (Passo 4).
# - Além disso, o único sinal considerado daqui em diante será o C4-A1.

# RESULTADOS OBTIDOS (BALANCETE DE DADOS):
# - Total de Sinal Processado: ~9,61 horas (17.725.440 amostras brutas).
# - Sinal Rejeitado: ~15 minutos (segmentos contendo drift inicial e artefatos).
# - Sinal Preservado: 9,36 horas (1.124 janelas 100% aprovadas pelos testes).
# - Estrutura Final: A matriz 'base_estatistica' foi gerada com dimensões 
#   (1124, 15360), representando dados purificados e prontos para extração de atributos.
# =============================================================================